# Sentinel weapon detector — gun / knife

**Runtime → Change runtime type → GPU** first.

## What changed and why

The previous model was trained on Sohas and is being discarded. Two measured
reasons:

| Problem | Measurement |
|---|---|
| Object scale | Sohas `pistol` median box = **43% of frame**. Guns in our footage ≈ **0.4%**. A 100x gap. |
| Synthetic composites | 228 pistols pasted at arbitrary positions — floating beside hips, lying on pavement. Taught "gun-shaped patch anywhere". |

Result on weapon-free footage: pistol fired on **7% of frames**, including
motorcycle fuel tanks, a car front at night, an umbrella, and a dark animal at
0.72 confidence.

## This run

| Source | Role |
|---|---|
| Roboflow project (gun + knife, polygons) | Positives. Polygons because a segmentation head must produce a plausible *silhouette* — much harder to fake on a fuel tank than a box. |
| `training/data/weapons/negatives` (1,100 frames) | Precision. Ugandan street footage, empty labels. Includes boda bodas, which caused the plurality of false pistols. |

Nothing trains until the scale audit passes.

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.116 onnx==1.22.0 roboflow kagglehub
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 1. Source data

Two routes. **Route A** is the intended one: the Kaggle set imported into
Roboflow and polygon-annotated there. **Route B** pulls the raw Kaggle boxes
directly, for a quick baseline before the labelling work is done.

Run one, not both.

In [ ]:
# ── Route A: Roboflow export (polygons) ─────────────────────────────────
# Workspace → project → Versions → Export → YOLOv8, "show download code".
from roboflow import Roboflow

RF_API_KEY   = ''      # keep out of git
RF_WORKSPACE = 'kwezi-louis'
RF_PROJECT   = 'sentinel-weapons'
RF_VERSION   = 2

rf = Roboflow(api_key=RF_API_KEY)
ds = (rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
        .version(RF_VERSION).download('yolov8'))
DATA_ROOT = ds.location
print('downloaded to', DATA_ROOT)

In [ ]:
# ── Route B: raw Kaggle boxes (baseline only) ───────────────────────────
# import kagglehub
# DATA_ROOT = kagglehub.dataset_download('raghavnanjappan/weapon-dataset-for-yolov5')
# print('downloaded to', DATA_ROOT)
# !find {DATA_ROOT} -maxdepth 3 -type d | head -30

## 2. Upload the hard negatives

Zip `training/data/weapons/negatives` on your Mac and upload it here:

```bash
cd training/data/weapons && zip -qr negatives.zip negatives
```

In [ ]:
from google.colab import files
import zipfile, pathlib

up = files.upload()                       # choose negatives.zip
zipfile.ZipFile(next(iter(up))).extractall('/content/mine')
NEG = pathlib.Path('/content/mine/negatives')
n_neg = len(list((NEG / 'images').glob('*.jpg')))
print('negatives:', n_neg)
assert n_neg, 'unexpected zip layout — expected negatives/images/*.jpg'

## 3. Audit before training

This is the gate that would have stopped the Sohas run. It measures object
area as a percentage of frame and compares it to the ~0.4% our footage needs.

Upload `training/weapons/audit_dataset.py` alongside, or paste it in.

In [ ]:
from google.colab import files
files.upload()          # choose audit_dataset.py
!python audit_dataset.py --root {DATA_ROOT} --target-pct 0.4

**Read the verdict before continuing.**

- `ok` — proceed.
- `Nx large` under 10x — proceed, but raise `scale=` augmentation below.
- `TOO LARGE` — do not simply train. The objects are posed close-ups and the
  model will learn size, not shape. Either find surveillance-distance footage
  or expect the same phantom-pistol failure as last time.

## 4. Merge negatives into train

Negatives go to **train only**. A val split containing negatives reports a
meaningless mAP, because there is nothing to detect in them.

In [ ]:
import shutil, pathlib, yaml

root = pathlib.Path(DATA_ROOT)
cfg  = yaml.safe_load((root / 'data.yaml').read_text())
names = cfg['names'] if isinstance(cfg['names'], list) else list(cfg['names'].values())
print('classes:', names)

tr_i = root / 'train' / 'images'
tr_l = root / 'train' / 'labels'
tr_i.mkdir(parents=True, exist_ok=True); tr_l.mkdir(parents=True, exist_ok=True)

added = 0
for img in sorted((NEG / 'images').glob('*.jpg')):
    shutil.copy(img, tr_i / f'neg_{img.name}')
    (tr_l / f'neg_{img.stem}.txt').write_text('')     # empty = 'nothing here'
    added += 1

n_train = len(list(tr_i.glob('*')))
print(f'added {added} negatives | train images now {n_train} '
      f'({100*added/n_train:.0f}% negative)')

# 10-30% negatives is the useful band. Below that they are drowned out; above
# it the model gets conservative and recall drops.
if added / n_train < 0.10:
    print('WARNING: under 10% negatives — mine more with mine_negatives.py')

cfg['path']  = str(root)
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images' if (root / 'valid').exists() else 'test/images'
(root / 'data.yaml').write_text(yaml.safe_dump(cfg))
print(cfg)

## 5. Train

`yolov8s-seg` for polygon labels; switch to `yolov8s` if your export is boxes.
Not `yolov8n` — the nano model is the one that called shrubs grenades.

`scale=0.9` is deliberate and is the single most important augmentation here:
it forces the model to see each object across a wide range of sizes, which is
the direct counter to a dataset shot closer than your cameras.

In [ ]:
from ultralytics import YOLO

SEG = True          # False if your Roboflow export is bounding boxes
m = YOLO('yolov8s-seg.pt' if SEG else 'yolov8s.pt')

m.train(data=str(root / 'data.yaml'),
        epochs=100, imgsz=960, batch=12, patience=25,
        scale=0.9, degrees=7, hsv_v=0.5, mosaic=1.0, close_mosaic=10,
        project='/content/runs', name='weapons_gk')

## 6. Export and download

In [ ]:
from ultralytics import YOLO
import json

best = '/content/runs/weapons_gk/weights/best.pt'
p = YOLO(best).export(format='onnx', imgsz=960, opset=12, simplify=False)
print('exported:', p)
print('\nSENTINEL_WEAPON_CLASSES=' + json.dumps(names))

from google.colab import files
files.download(p)

## 7. The only number that matters

Colab's mAP is measured on the distribution the model was fitted on. It will
look good. Ignore it.

On your Mac:

```bash
cp ~/Downloads/best.onnx checkpoints/weapons_v2.onnx
python -m training.weapons.evaluate \
    --model checkpoints/weapons_v2.onnx \
    --classes gun knife
```

That reports **false alarms per 100 weapon-free frames**. The previous model
sat at roughly 7 per 100 at conf 0.25, on footage containing no weapons at all.

Only if it passes:

```bash
export SENTINEL_WEAPON_MODEL=checkpoints/weapons_v2.onnx
export SENTINEL_WEAPON_CLASSES='["gun","knife"]'
```